# 🏆 DATATHON 2026 — GenCore | Revenue & COGS Forecasting
**XGBoost + LightGBM Ensemble | Recursive Forecast | 62 Features**

In [ ]:
import os, warnings, gc, glob
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

KAGGLE = os.path.exists('/kaggle/input')
if KAGGLE:
    # Find sales.csv anywhere under /kaggle/input/
    matches = glob.glob('/kaggle/input/**/sales.csv', recursive=True)
    if not matches:
        # Debug: print directory structure
        for root, dirs, files in os.walk('/kaggle/input'):
            for f in files[:5]:
                print(f"  {os.path.join(root, f)}")
        raise FileNotFoundError('sales.csv not found under /kaggle/input/')
    DATA_DIR = os.path.dirname(matches[0])
    OUT_DIR  = '/kaggle/working'
else:
    DATA_DIR = 'data/raw'
    OUT_DIR  = 'output'

TARGETS = ['Revenue', 'COGS']
LAGS = [7, 14, 21, 28]
ROLL_W = [7, 14, 28]
print(f"ENV = {'Kaggle' if KAGGLE else 'Local'} | DATA = {DATA_DIR}")

In [ ]:
sales = pd.read_csv(f'{DATA_DIR}/sales.csv')
sales['Date'] = pd.to_datetime(sales['Date'])
sales = sales.sort_values('Date').reset_index(drop=True)

sub_tpl = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
sub_tpl['Date'] = pd.to_datetime(sub_tpl['Date'])
forecast_dates = sub_tpl['Date'].tolist()

print(f"Train : {sales.Date.min().date()} → {sales.Date.max().date()} ({len(sales)} rows)")
print(f"Fcast : {forecast_dates[0].date()} → {forecast_dates[-1].date()} ({len(forecast_dates)} rows)")

# ── Web traffic ──
wt = pd.read_csv(f'{DATA_DIR}/web_traffic.csv')
wt['date'] = pd.to_datetime(wt['date'])
wt_num = [c for c in ['sessions','unique_visitors','page_views','bounce_rate','avg_session_duration_sec'] if c in wt.columns]
wt_daily = wt.groupby('date', as_index=False)[wt_num].mean()

# ── Transaction daily stats ──
orders = pd.read_csv(f'{DATA_DIR}/orders.csv')
orders['order_date'] = pd.to_datetime(orders['order_date'])
oi = pd.read_csv(f'{DATA_DIR}/order_items.csv')
ret = pd.read_csv(f'{DATA_DIR}/returns.csv')
ret['return_date'] = pd.to_datetime(ret['return_date'])

d_ord = orders.groupby('order_date').agg(
    daily_orders=('order_id','nunique')
).reset_index().rename(columns={'order_date':'Date'})

oi2 = oi.merge(orders[['order_id','order_date']], on='order_id', how='left')
d_items = oi2.groupby('order_date').agg(
    daily_items=('product_id','count'),
    daily_discount=('discount_amount','sum'),
).reset_index().rename(columns={'order_date':'Date'})
d_items['Date'] = pd.to_datetime(d_items['Date'])

d_ret = ret.groupby('return_date').agg(
    daily_returns=('return_id','count'),
).reset_index().rename(columns={'return_date':'Date'})
d_ret['Date'] = pd.to_datetime(d_ret['Date'])

# ── Inventory monthly ──
inv = pd.read_csv(f'{DATA_DIR}/inventory.csv')
inv['snapshot_date'] = pd.to_datetime(inv['snapshot_date'])
inv_cols = [c for c in ['stock_on_hand','units_sold','fill_rate','sell_through_rate','reorder_flag'] if c in inv.columns]
inv['_y'] = inv.snapshot_date.dt.year
inv['_m'] = inv.snapshot_date.dt.month
inv_mo = inv.groupby(['_y','_m'], as_index=False)[inv_cols].mean()

print("Auxiliary data loaded ✓")
del oi, oi2, wt; gc.collect()

In [ ]:

def add_calendar(df):
    d = df['Date']
    df['year']         = d.dt.year
    df['month']        = d.dt.month
    df['day']          = d.dt.day
    df['dayofweek']    = d.dt.dayofweek
    df['dayofyear']    = d.dt.dayofyear
    df['weekofyear']   = d.dt.isocalendar().week.astype(int)
    df['quarter']      = d.dt.quarter
    df['is_weekend']   = (df['dayofweek'] >= 5).astype(int)
    df['is_month_start'] = d.dt.is_month_start.astype(int)
    df['is_month_end']   = d.dt.is_month_end.astype(int)
    df['is_quarter_end'] = d.dt.is_quarter_end.astype(int)
    md = d.dt.strftime('%m-%d')
    df['is_vn_holiday'] = md.isin({'01-01','04-30','05-01','09-02'}).astype(int)
    df['is_tet']        = df['month'].isin([1,2]).astype(int)
    df['is_payday']     = df['day'].isin([1,15]).astype(int)
    return df

def add_fourier(df):
    for k in [1,2,3]:
        df[f'sin_y{k}'] = np.sin(2*np.pi*k*df['dayofyear']/365.25)
        df[f'cos_y{k}'] = np.cos(2*np.pi*k*df['dayofyear']/365.25)
    for k in [1,2]:
        df[f'sin_w{k}'] = np.sin(2*np.pi*k*df['dayofweek']/7)
        df[f'cos_w{k}'] = np.cos(2*np.pi*k*df['dayofweek']/7)
    return df

def add_lags_rolling(df):
    df = df.sort_values('Date').copy()
    for t in TARGETS:
        if t not in df.columns:
            continue
        s = df[t]
        for lag in LAGS:
            df[f'{t}_lag{lag}'] = s.shift(lag)
        shifted = s.shift(1)
        for w in ROLL_W:
            df[f'{t}_rmean{w}'] = shifted.rolling(w, min_periods=1).mean()
            df[f'{t}_rstd{w}']  = shifted.rolling(w, min_periods=1).std().fillna(0)
        df[f'{t}_ewm7']  = shifted.ewm(span=7, min_periods=1).mean()
        df[f'{t}_ewm28'] = shifted.ewm(span=28, min_periods=1).mean()
    return df

def merge_externals(df):
    df = df.merge(wt_daily, left_on='Date', right_on='date', how='left')
    if 'date' in df.columns: df.drop(columns='date', inplace=True)
    df = df.merge(d_ord, on='Date', how='left')
    df = df.merge(d_items, on='Date', how='left')
    df = df.merge(d_ret, on='Date', how='left')
    df['_y'] = df['Date'].dt.year
    df['_m'] = df['Date'].dt.month
    df = df.merge(inv_mo, on=['_y','_m'], how='left')
    df.drop(columns=['_y','_m'], inplace=True)
    for c in df.columns:
        if c not in ['Date'] + TARGETS and df[c].isna().any():
            df[c] = df[c].ffill().bfill().fillna(0)
    return df

def build_full_features(df):
    df = add_calendar(df)
    df = add_fourier(df)
    df = add_lags_rolling(df)
    df = merge_externals(df)
    return df

In [ ]:
train = build_full_features(sales.copy())

lag_cols = [c for c in train.columns if '_lag' in c]
train = train.dropna(subset=lag_cols).reset_index(drop=True)

feat_cols = sorted([c for c in train.columns if c not in ['Date'] + TARGETS])
print(f"Training set: {train.shape[0]} rows × {len(feat_cols)} features")

X_train = train[feat_cols]
y_rev   = train['Revenue']
y_cogs  = train['COGS']

In [ ]:
def cv_evaluate(X, y, model_fn, n_splits=5, label=''):
    tss = TimeSeriesSplit(n_splits=n_splits)
    maes, rmses, r2s = [], [], []
    for fold, (tr_idx, va_idx) in enumerate(tss.split(X)):
        mdl = model_fn()
        mdl.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        p = mdl.predict(X.iloc[va_idx])
        maes.append(mean_absolute_error(y.iloc[va_idx], p))
        rmses.append(np.sqrt(mean_squared_error(y.iloc[va_idx], p)))
        r2s.append(r2_score(y.iloc[va_idx], p))
    print(f"  {label:18s} | MAE={np.mean(maes):>12,.0f} | RMSE={np.mean(rmses):>12,.0f} | R²={np.mean(r2s):.4f}")
    return {'MAE': np.mean(maes), 'RMSE': np.mean(rmses), 'R2': np.mean(r2s)}

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

def make_xgb():
    return XGBRegressor(
        n_estimators=1200, learning_rate=0.012, max_depth=7,
        subsample=0.85, colsample_bytree=0.85, min_child_weight=3,
        reg_alpha=0.01, reg_lambda=0.1, random_state=SEED, n_jobs=-1,
    )
def make_lgbm():
    return LGBMRegressor(
        n_estimators=1200, learning_rate=0.012, num_leaves=40, max_depth=7,
        subsample=0.85, colsample_bytree=0.85,
        reg_alpha=0.01, reg_lambda=0.1,
        random_state=SEED, n_jobs=-1, verbosity=-1,
    )

print("=== Revenue CV ===")
cv_evaluate(X_train, y_rev, make_xgb,  label='XGBoost Revenue')
cv_evaluate(X_train, y_rev, make_lgbm, label='LightGBM Revenue')
print("\n=== COGS CV ===")
cv_evaluate(X_train, y_cogs, make_xgb,  label='XGBoost COGS')
cv_evaluate(X_train, y_cogs, make_lgbm, label='LightGBM COGS')

In [ ]:
print("Training final models on all data ...")
xgb_rev = make_xgb(); xgb_rev.fit(X_train, y_rev)
lgb_rev = make_lgbm(); lgb_rev.fit(X_train, y_rev)
xgb_cogs = make_xgb(); xgb_cogs.fit(X_train, y_cogs)
lgb_cogs = make_lgbm(); lgb_cogs.fit(X_train, y_cogs)
print("4 models trained ✓")

imp = pd.Series(xgb_rev.feature_importances_, index=feat_cols).sort_values(ascending=False)
print("\nTop 15 features (XGBoost Revenue):")
print(imp.head(15).to_string())

In [ ]:
print(f"\nRecursive forecasting {len(forecast_dates)} days ...")

HIST_WINDOW = 120
history = sales[['Date','Revenue','COGS']].copy().sort_values('Date').tail(HIST_WINDOW).reset_index(drop=True)

ext_wt = wt_daily.set_index('date')
ext_ord = d_ord.set_index('Date')
ext_items = d_items.set_index('Date')
ext_ret = d_ret.set_index('Date')
ext_inv = inv_mo.set_index(['_y','_m'])

def _build_single_day_features(hist, fdate):
    row = {}
    dt = pd.Timestamp(fdate)
    row['year'] = dt.year; row['month'] = dt.month; row['day'] = dt.day
    row['dayofweek'] = dt.dayofweek; row['dayofyear'] = dt.dayofyear
    row['weekofyear'] = dt.isocalendar().week; row['quarter'] = dt.quarter
    row['is_weekend'] = int(dt.dayofweek >= 5)
    row['is_month_start'] = int(dt.is_month_start); row['is_month_end'] = int(dt.is_month_end)
    row['is_quarter_end'] = int(dt.is_quarter_end)
    md = dt.strftime('%m-%d')
    row['is_vn_holiday'] = int(md in {'01-01','04-30','05-01','09-02'})
    row['is_tet'] = int(dt.month in [1,2]); row['is_payday'] = int(dt.day in [1,15])
    for k in [1,2,3]:
        row[f'sin_y{k}'] = np.sin(2*np.pi*k*dt.dayofyear/365.25)
        row[f'cos_y{k}'] = np.cos(2*np.pi*k*dt.dayofyear/365.25)
    for k in [1,2]:
        row[f'sin_w{k}'] = np.sin(2*np.pi*k*dt.dayofweek/7)
        row[f'cos_w{k}'] = np.cos(2*np.pi*k*dt.dayofweek/7)
    rev_arr = hist['Revenue'].values
    cogs_arr = hist['COGS'].values
    for t, arr in [('Revenue', rev_arr), ('COGS', cogs_arr)]:
        for lag in LAGS:
            row[f'{t}_lag{lag}'] = arr[-lag] if len(arr) >= lag else arr[0]
        for w in ROLL_W:
            window = arr[-w:] if len(arr) >= w else arr
            row[f'{t}_rmean{w}'] = np.mean(window)
            row[f'{t}_rstd{w}']  = np.std(window) if len(window) > 1 else 0.0
        row[f'{t}_ewm7']  = pd.Series(arr).ewm(span=7, min_periods=1).mean().iloc[-1]
        row[f'{t}_ewm28'] = pd.Series(arr).ewm(span=28, min_periods=1).mean().iloc[-1]
    if fdate in ext_wt.index:
        for c in wt_num: row[c] = ext_wt.at[fdate, c]
    if fdate in ext_ord.index: row['daily_orders'] = ext_ord.at[fdate, 'daily_orders']
    if fdate in ext_items.index:
        row['daily_items'] = ext_items.at[fdate, 'daily_items']
        row['daily_discount'] = ext_items.at[fdate, 'daily_discount']
    if fdate in ext_ret.index: row['daily_returns'] = ext_ret.at[fdate, 'daily_returns']
    inv_key = (dt.year, dt.month)
    if inv_key in ext_inv.index:
        for c in inv_cols: row[c] = ext_inv.at[inv_key, c]
    return row

last_known = {}
for c in feat_cols:
    if c in train.columns:
        last_known[c] = float(train[c].iloc[-1])

predictions = []
for i, fdate in enumerate(forecast_dates):
    row = _build_single_day_features(history, fdate)
    for c in feat_cols:
        if c not in row or (isinstance(row.get(c), float) and np.isnan(row[c])):
            row[c] = last_known.get(c, 0.0)
        else:
            last_known[c] = float(row[c])
    X_f = np.array([[row[c] for c in feat_cols]])
    rev_pred  = max(0.0, 0.5 * xgb_rev.predict(X_f)[0] + 0.5 * lgb_rev.predict(X_f)[0])
    cogs_pred = max(0.0, 0.5 * xgb_cogs.predict(X_f)[0] + 0.5 * lgb_cogs.predict(X_f)[0])
    predictions.append({'Date': fdate, 'Revenue': rev_pred, 'COGS': cogs_pred})
    new_row = pd.DataFrame([{'Date': fdate, 'Revenue': rev_pred, 'COGS': cogs_pred}])
    history = pd.concat([history, new_row], ignore_index=True).tail(HIST_WINDOW).reset_index(drop=True)
    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(forecast_dates)} done | Rev={rev_pred:,.0f} | COGS={cogs_pred:,.0f}")

pred_df = pd.DataFrame(predictions)
print(f"Recursive forecast complete: {pred_df.shape}")

In [ ]:
submission = sub_tpl[['Date']].copy()
submission = submission.merge(pred_df, on='Date', how='left')
submission['Revenue'] = submission['Revenue'].clip(lower=0).round(2)
submission['COGS']    = submission['COGS'].clip(lower=0).round(2)
submission['Date']    = submission['Date'].dt.strftime('%Y-%m-%d')

os.makedirs(OUT_DIR, exist_ok=True)
out_path = f'{OUT_DIR}/submission.csv'
submission.to_csv(out_path, index=False, float_format='%.2f')
print(f"\n✅ Submission saved: {out_path}")
print(f"   Rows: {len(submission)}")
print(f"   Revenue: mean={submission.Revenue.mean():,.0f}  std={submission.Revenue.std():,.0f}")
print(f"   COGS:    mean={submission.COGS.mean():,.0f}  std={submission.COGS.std():,.0f}")
print(submission.head(10))

In [ ]:
assert len(submission) == len(sub_tpl), f"Row count mismatch: {len(submission)} vs {len(sub_tpl)}"
assert submission.Revenue.isna().sum() == 0, "NaN in Revenue!"
assert submission.COGS.isna().sum() == 0, "NaN in COGS!"
assert (submission.Revenue >= 0).all(), "Negative Revenue!"
assert (submission.COGS >= 0).all(), "Negative COGS!"
print("\n✅ All sanity checks passed!")